# TIL-AI YOLOv8s 960 Robust Fine-Tuning Notebook

Run this notebook from top to bottom.

What it does:
1. Loads the original COCO-style dataset.
2. Converts labels to YOLO format.
3. Creates train/validation split.
4. Extracts your saved `yolov8s_960.zip` and recovers `weights/best.pt`.
5. Builds robust/noisy training and validation datasets.
6. Fine-tunes `yolov8s_960` at image size 960.
7. Validates on clean and noisy validation images.
8. Sweeps confidence thresholds.
9. Exports the final robust model.

Upload `yolov8s_960.zip` to the trainer workspace before running. If the zip is in another path, edit `MODEL_ZIP_CANDIDATES` in the setup cell.


## 1. Install and import dependencies


In [1]:
!pip install -q ultralytics iterative-stratification

from pathlib import Path
from collections import Counter, defaultdict
import json
import os
import random
import shutil
import time
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from tqdm.auto import tqdm
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports ready")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.7 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Imports ready
Torch: 2.10.0+cu128
CUDA available: True


## 2. Configure paths

If your dataset path is different, change `BASE_PATH` here.


In [2]:
# Dataset path used by the previous trainer notebook
BASE_PATH = Path("/kaggle/input/datasets/wongpohyee/advancedcv/cv")
IMAGES_DIR = BASE_PATH / "images"
ANNOTATION_PATH = BASE_PATH / "annotations.json"

# Working/output path
OUTPUT_PATH = Path("/kaggle/working/yolo")
LABELS_DIR = OUTPUT_PATH / "labels"
TRAIN_IMG_DIR = OUTPUT_PATH / "train/images"
VAL_IMG_DIR = OUTPUT_PATH / "val/images"
TRAIN_LBL_DIR = OUTPUT_PATH / "train/labels"
VAL_LBL_DIR = OUTPUT_PATH / "val/labels"
DATA_YAML = OUTPUT_PATH / "data.yaml"

ROBUST_TRAIN_IMG_DIR = OUTPUT_PATH / "train_robust/images"
ROBUST_TRAIN_LBL_DIR = OUTPUT_PATH / "train_robust/labels"
ROBUST_VAL_IMG_DIR = OUTPUT_PATH / "val_robust/images"
ROBUST_VAL_LBL_DIR = OUTPUT_PATH / "val_robust/labels"
DATA_ROBUST_YAML = OUTPUT_PATH / "data_robust.yaml"
DATA_ROBUST_VAL_YAML = OUTPUT_PATH / "data_robust_val.yaml"

RUNS_DIR = Path("/kaggle/runs/detect")
PREDICT_DIR = Path("/kaggle/predict")
BACKUP_DIR = Path("/kaggle/backups")
EXTRACT_DIR = Path("/kaggle/extracted_models")

# Put/upload yolov8s_960.zip in one of these locations.
MODEL_ZIP_CANDIDATES = [
    Path("/workspace/yolov8s_960.zip"),
    Path("/workspace/input/yolov8s_960.zip"),
    Path("/kaggle/input/yolov8s-960/yolov8s_960.zip"),
    Path("/content/yolov8s_960.zip"),
]

for d in [
    OUTPUT_PATH, LABELS_DIR, TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_LBL_DIR, VAL_LBL_DIR,
    ROBUST_TRAIN_IMG_DIR, ROBUST_TRAIN_LBL_DIR, ROBUST_VAL_IMG_DIR, ROBUST_VAL_LBL_DIR,
    RUNS_DIR, PREDICT_DIR, BACKUP_DIR, EXTRACT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_PATH exists:", BASE_PATH.exists())
print("IMAGES_DIR exists:", IMAGES_DIR.exists())
print("ANNOTATION_PATH exists:", ANNOTATION_PATH.exists())
print("OUTPUT_PATH:", OUTPUT_PATH)


BASE_PATH exists: True
IMAGES_DIR exists: True
ANNOTATION_PATH exists: True
OUTPUT_PATH: /kaggle/working/yolo


## 3. Load annotations and class names


In [3]:
if not ANNOTATION_PATH.exists():
    raise FileNotFoundError(f"annotations.json not found at {ANNOTATION_PATH}. Update BASE_PATH in the previous cell.")

with open(ANNOTATION_PATH, "r") as f:
    data = json.load(f)

categories = sorted(data["categories"], key=lambda x: x["id"])
cat_map = {c["id"]: c["name"] for c in categories}
cat_id_to_idx = {cat_id: idx for idx, cat_id in enumerate(cat_map.keys())}
class_names = [cat_map[cat_id] for cat_id in cat_map.keys()]

print("Images:", len(data["images"]))
print("Annotations:", len(data["annotations"]))
print("Classes:", len(class_names))
for i, name in enumerate(class_names):
    print(i, name)

counts = Counter(ann["category_id"] for ann in data["annotations"])
df_class_counts = pd.DataFrame([
    {
        "category_id": cat_id,
        "yolo_class_id": cat_id_to_idx[cat_id],
        "class_name": cat_map[cat_id],
        "annotation_count": count,
    }
    for cat_id, count in sorted(counts.items())
])
df_class_counts


Images: 5000
Annotations: 19392
Classes: 18
0 cargo aircraft
1 commercial aircraft
2 drone
3 fighter jet
4 fighter plane
5 helicopter
6 light aircraft
7 missile
8 truck
9 car
10 tank
11 bus
12 van
13 cargo ship
14 yacht
15 cruise ship
16 warship
17 sailboat


,category_id,yolo_class_id,class_name,annotation_count
0,0,0,cargo aircraft,292
1,1,1,commercial aircraft,1703
2,2,2,drone,941
3,3,3,fighter jet,2515
4,4,4,fighter plane,1207
5,5,5,helicopter,1597
6,6,6,light aircraft,910
7,7,7,missile,805
8,8,8,truck,2353
9,9,9,car,983


## 4. Convert COCO annotations to YOLO labels


In [4]:
image_to_anns = defaultdict(list)
for ann in data["annotations"]:
    image_to_anns[ann["image_id"]].append(ann)

for img_info in tqdm(data["images"], desc="Converting labels"):
    img_id = img_info["id"]
    img_w = img_info["width"]
    img_h = img_info["height"]
    stem = Path(img_info["file_name"]).stem
    label_path = LABELS_DIR / f"{stem}.txt"

    lines = []
    for ann in image_to_anns[img_id]:
        x, y, w, h = ann["bbox"]
        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        norm_w = w / img_w
        norm_h = h / img_h
        class_id = cat_id_to_idx[ann["category_id"]]
        lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")

    label_path.write_text("\n".join(lines))

print("Labels saved to:", LABELS_DIR)
print("Label files:", len(list(LABELS_DIR.glob("*.txt"))))


Converting labels:   0%|          | 0/5000 [00:00<?, ?it/s]

Labels saved to: /kaggle/working/yolo/labels
Label files: 5000


## 5. Create train/validation split

This uses multilabel stratification so each class is represented more evenly.


In [5]:
# Clean old split folders so reruns are consistent
for d in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_LBL_DIR, VAL_LBL_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

images = sorted(
    list(IMAGES_DIR.glob("*.jpg")) +
    list(IMAGES_DIR.glob("*.jpeg")) +
    list(IMAGES_DIR.glob("*.png"))
)

image_paths = []
multi_labels = []

for img_path in images:
    label_path = LABELS_DIR / f"{img_path.stem}.txt"
    if not label_path.exists():
        continue

    class_ids = set()
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_ids.add(int(parts[0]))

    label_vector = [0] * len(class_names)
    for cls_id in class_ids:
        label_vector[cls_id] = 1

    image_paths.append(img_path)
    multi_labels.append(label_vector)

X = np.array(image_paths)
y = np.array(multi_labels)

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(msss.split(X, y))
train_images = X[train_idx]
val_images = X[val_idx]

def copy_image_and_label(img_path, img_dst_dir, lbl_dst_dir):
    dst_img = img_dst_dir / img_path.name
    label_src = LABELS_DIR / f"{img_path.stem}.txt"
    label_dst = lbl_dst_dir / f"{img_path.stem}.txt"

    if not dst_img.exists():
        shutil.copy2(img_path, dst_img)
    if label_src.exists() and not label_dst.exists():
        shutil.copy2(label_src, label_dst)

for img_path in tqdm(train_images, desc="Copy train"):
    copy_image_and_label(img_path, TRAIN_IMG_DIR, TRAIN_LBL_DIR)

for img_path in tqdm(val_images, desc="Copy val"):
    copy_image_and_label(img_path, VAL_IMG_DIR, VAL_LBL_DIR)

print("Train images:", len(list(TRAIN_IMG_DIR.glob("*"))))
print("Val images:", len(list(VAL_IMG_DIR.glob("*"))))


Copy train:   0%|          | 0/4000 [00:00<?, ?it/s]

Copy val:   0%|          | 0/1000 [00:00<?, ?it/s]

Train images: 4000
Val images: 1000


## 6. Create YOLO data YAML


In [6]:
yaml_content = {
    "path": str(OUTPUT_PATH),
    "train": "train/images",
    "val": "val/images",
    "nc": len(class_names),
    "names": class_names,
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(yaml_content, f, sort_keys=False)

print(DATA_YAML.read_text())


path: /kaggle/working/yolo
train: train/images
val: val/images
nc: 18
names:
- cargo aircraft
- commercial aircraft
- drone
- fighter jet
- fighter plane
- helicopter
- light aircraft
- missile
- truck
- car
- tank
- bus
- van
- cargo ship
- yacht
- cruise ship
- warship
- sailboat



## 7. Extract your saved yolov8s_960.zip

This recovers `weights/best.pt` from the zip. If the notebook cannot find the zip, upload it to `/workspace/yolov8s_960.zip` or edit `MODEL_ZIP_CANDIDATES` above.


In [8]:
# Load saved yolov8s_960 model directly 

SAVED_MODEL_DIR = Path("/kaggle/input/datasets/wongpohyee/yolov8s-960/yolov8s_960")
BASE_MODEL_PATH = SAVED_MODEL_DIR / "weights/best.pt"

if not BASE_MODEL_PATH.exists():
    print("Could not find:", BASE_MODEL_PATH)
    print("Files inside saved model directory:")

    if SAVED_MODEL_DIR.exists():
        for p in sorted(SAVED_MODEL_DIR.rglob("*")):
            print("-", p)
    else:
        print("Directory does not exist:", SAVED_MODEL_DIR)

    raise FileNotFoundError(BASE_MODEL_PATH)

BASE_MODEL_PATH = str(BASE_MODEL_PATH)
print("Using saved base model:", BASE_MODEL_PATH)

Using saved base model: /kaggle/input/datasets/wongpohyee/yolov8s-960/yolov8s_960/weights/best.pt


## 8. GPU setup


In [9]:
GPU_DEVICE = 0
TRAIN_BATCH = 0.90
TRAIN_WORKERS = min(16, max(2, (os.cpu_count() or 2) - 2))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available. Enable GPU before training.")

torch.cuda.set_device(GPU_DEVICE)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

gpu = torch.cuda.get_device_properties(GPU_DEVICE)
print(f"Using GPU {GPU_DEVICE}: {gpu.name} ({gpu.total_memory / 1024**3:.1f} GB VRAM)")
print("AutoBatch target:", TRAIN_BATCH)
print("Workers:", TRAIN_WORKERS)


Using GPU 0: Tesla T4 (14.6 GB VRAM)
AutoBatch target: 0.9
Workers: 2


## 9. Define robust corruption functions

These simulate unknown challenge noise: Gaussian noise, compression, blur, brightness/contrast changes, resize artifacts, and sharpening.


In [12]:
def gaussian_noise(img, std=10):
    noise = np.random.normal(0, std, img.shape).astype(np.float32)
    out = img.astype(np.float32) + noise
    return np.clip(out, 0, 255).astype(np.uint8)

def brightness_contrast(img, alpha=1.15, beta=8):
    out = img.astype(np.float32) * alpha + beta
    return np.clip(out, 0, 255).astype(np.uint8)

def jpeg_compress(img, quality=45):
    ok, enc = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    if not ok:
        return img
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)

def blur(img, k=3):
    return cv2.GaussianBlur(img, (k, k), 0)

def downscale_upscale(img, scale=0.65):
    h, w = img.shape[:2]
    small_w = max(1, int(w * scale))
    small_h = max(1, int(h * scale))
    small = cv2.resize(img, (small_w, small_h), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

def sharpen(img):
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)

def random_corruption(img):
    funcs = [
        lambda x: gaussian_noise(x, std=np.random.choice([5, 8, 12, 16, 20])),
        lambda x: brightness_contrast(x, alpha=np.random.uniform(0.70, 1.40), beta=np.random.randint(-22, 23)),
        lambda x: jpeg_compress(x, quality=np.random.choice([30, 40, 50, 60, 75])),
        lambda x: blur(x, k=3),
        lambda x: downscale_upscale(x, scale=np.random.choice([0.50, 0.65, 0.80])),
        sharpen,
    ]

    out = img.copy()
    chosen = np.random.choice(funcs, size=np.random.choice([1, 2, 3]), replace=False)
    for f in chosen:
        out = f(out)
    return out


## 10. Build robust train and validation datasets

Labels are copied unchanged because corruption does not move objects.


In [13]:
# Increase to 3 if you have enough GPU time. Start with 2 for a good balance.
ROBUST_COPIES_PER_IMAGE = 2

for d in [ROBUST_TRAIN_IMG_DIR, ROBUST_TRAIN_LBL_DIR, ROBUST_VAL_IMG_DIR, ROBUST_VAL_LBL_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

# Original clean train images
for img_path in tqdm(sorted(TRAIN_IMG_DIR.glob("*")), desc="Copy clean train"):
    lbl_path = TRAIN_LBL_DIR / f"{img_path.stem}.txt"
    if not lbl_path.exists():
        continue
    shutil.copy2(img_path, ROBUST_TRAIN_IMG_DIR / img_path.name)
    shutil.copy2(lbl_path, ROBUST_TRAIN_LBL_DIR / lbl_path.name)

# Corrupted train copies
for img_path in tqdm(sorted(TRAIN_IMG_DIR.glob("*")), desc="Create robust train"):
    lbl_path = TRAIN_LBL_DIR / f"{img_path.stem}.txt"
    if not lbl_path.exists():
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue

    for i in range(ROBUST_COPIES_PER_IMAGE):
        aug = random_corruption(img)
        aug_name = f"{img_path.stem}_robust{i}{img_path.suffix}"
        aug_label = f"{img_path.stem}_robust{i}.txt"
        cv2.imwrite(str(ROBUST_TRAIN_IMG_DIR / aug_name), aug)
        shutil.copy2(lbl_path, ROBUST_TRAIN_LBL_DIR / aug_label)

# One corrupted validation version for robustness selection
for img_path in tqdm(sorted(VAL_IMG_DIR.glob("*")), desc="Create robust val"):
    lbl_path = VAL_LBL_DIR / f"{img_path.stem}.txt"
    if not lbl_path.exists():
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue

    aug = random_corruption(img)
    cv2.imwrite(str(ROBUST_VAL_IMG_DIR / img_path.name), aug)
    shutil.copy2(lbl_path, ROBUST_VAL_LBL_DIR / lbl_path.name)

print("Robust train images:", len(list(ROBUST_TRAIN_IMG_DIR.glob("*"))))
print("Robust train labels:", len(list(ROBUST_TRAIN_LBL_DIR.glob("*"))))
print("Robust val images:", len(list(ROBUST_VAL_IMG_DIR.glob("*"))))
print("Robust val labels:", len(list(ROBUST_VAL_LBL_DIR.glob("*"))))


Copy clean train:   0%|          | 0/4000 [00:00<?, ?it/s]

Create robust train:   0%|          | 0/4000 [00:00<?, ?it/s]

Create robust val:   0%|          | 0/1000 [00:00<?, ?it/s]

Robust train images: 12000
Robust train labels: 12000
Robust val images: 1000
Robust val labels: 1000


## 11. Create robust YOLO YAML files


In [14]:
yaml_content_robust = {
    "path": str(OUTPUT_PATH),
    "train": "train_robust/images",
    "val": "val/images",
    "nc": len(class_names),
    "names": class_names,
}

with open(DATA_ROBUST_YAML, "w") as f:
    yaml.safe_dump(yaml_content_robust, f, sort_keys=False)

yaml_content_robust_val = {
    "path": str(OUTPUT_PATH),
    "train": "train_robust/images",
    "val": "val_robust/images",
    "nc": len(class_names),
    "names": class_names,
}

with open(DATA_ROBUST_VAL_YAML, "w") as f:
    yaml.safe_dump(yaml_content_robust_val, f, sort_keys=False)

print("Training YAML:")
print(DATA_ROBUST_YAML.read_text())
print("Noisy validation YAML:")
print(DATA_ROBUST_VAL_YAML.read_text())


Training YAML:
path: /kaggle/working/yolo
train: train_robust/images
val: val/images
nc: 18
names:
- cargo aircraft
- commercial aircraft
- drone
- fighter jet
- fighter plane
- helicopter
- light aircraft
- missile
- truck
- car
- tank
- bus
- van
- cargo ship
- yacht
- cruise ship
- warship
- sailboat

Noisy validation YAML:
path: /kaggle/working/yolo
train: train_robust/images
val: val_robust/images
nc: 18
names:
- cargo aircraft
- commercial aircraft
- drone
- fighter jet
- fighter plane
- helicopter
- light aircraft
- missile
- truck
- car
- tank
- bus
- van
- cargo ship
- yacht
- cruise ship
- warship
- sailboat



## 12. Fine-tune yolov8s_960 robust model

This trains from your extracted `best.pt`. It keeps image size at 960, exactly as requested.


In [15]:
robust_model = YOLO(BASE_MODEL_PATH)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(GPU_DEVICE)

train_result = robust_model.train(
    data=str(DATA_ROBUST_YAML),
    epochs=40,
    imgsz=960,
    batch=TRAIN_BATCH,
    patience=12,
    cache="ram",
    project=str(RUNS_DIR),
    name="yolov8s_960_robust",
    exist_ok=True,
    device=GPU_DEVICE,
    workers=TRAIN_WORKERS,
    amp=True,
    mosaic=0.7,
    close_mosaic=8,
    mixup=0.05,
    copy_paste=0.05,
    hsv_h=0.02,
    hsv_s=0.6,
    hsv_v=0.5,
    degrees=3.0,
    translate=0.12,
    scale=0.6,
    fliplr=0.5,
    erasing=0.25,
    verbose=True,
)

peak_allocated_gb = torch.cuda.max_memory_allocated(GPU_DEVICE) / 1024**3
peak_reserved_gb = torch.cuda.max_memory_reserved(GPU_DEVICE) / 1024**3
print(f"GPU peak memory allocated: {peak_allocated_gb:.2f} GB | reserved: {peak_reserved_gb:.2f} GB")


Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=0.9, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=8, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.05, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo/data_robust.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.25, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/kaggle/input/datasets/wongpohyee/yolov8s-960/yolov8s_960/weights/best.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=yolov8s_960_robust, nbs=64, nms=Fal

KeyboardInterrupt: 

In [ ]:
# salvaging error

from pathlib import Path
import pandas as pd

RUN_DIR = Path("/kaggle/runs/detect/yolov8s_960_robust")
LAST_PT = RUN_DIR / "weights/last.pt"
BEST_PT = RUN_DIR / "weights/best.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

print("Run dir exists:", RUN_DIR.exists())
print("last.pt exists:", LAST_PT.exists())
print("best.pt exists:", BEST_PT.exists())
print("results.csv exists:", RESULTS_CSV.exists())

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    print("Completed epochs:", len(df))
    display(df.tail())

from ultralytics import YOLO

resume_model = YOLO(str(LAST_PT))
resume_model.train(resume=True)

from ultralytics import YOLO

resume_model = YOLO(str(BEST_PT))

resume_model.train(
    data=str(DATA_ROBUST_YAML),
    epochs=20,   # continue for 20 more epochs, not 40 again
    imgsz=960,
    batch=TRAIN_BATCH,
    patience=8,
    cache="ram",
    project=str(RUNS_DIR),
    name="yolov8s_960_robust_continue",
    exist_ok=True,
    device=GPU_DEVICE,
    workers=TRAIN_WORKERS,
    amp=True,
    mosaic=0.7,
    close_mosaic=8,
    mixup=0.05,
    copy_paste=0.05,
    hsv_h=0.02,
    hsv_s=0.6,
    hsv_v=0.5,
    degrees=3.0,
    translate=0.12,
    scale=0.6,
    fliplr=0.5,
    erasing=0.25,
    verbose=True,
)

ROBUST_BEST_PATH = Path("/kaggle/runs/detect/yolov8s_960_robust/weights/best.pt")
CROBUST_BEST_PATH = Path("/kaggle/runs/detect/yolov8s_960_robust_continue/weights/best.pt")

Run dir exists: True
last.pt exists: True
best.pt exists: True
results.csv exists: True
Completed epochs: 19


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
14,15,6738.97,0.49538,0.34930,0.83855,0.98643,0.98144,0.99195,0.93254,0.33826,0.24130,0.79870,0.000297,0.000297,0.000297
15,16,7184.44,0.49024,0.34310,0.83956,0.99055,0.97724,0.99209,0.93674,0.32528,0.23637,0.79711,0.000286,0.000286,0.000286
16,17,7630.24,0.48289,0.33700,0.83534,0.98580,0.98017,0.99178,0.93613,0.31528,0.22951,0.79331,0.000275,0.000275,0.000275
17,18,8074.83,0.47621,0.32667,0.83256,0.98210,0.97733,0.99191,0.94122,0.30792,0.22736,0.79147,0.000264,0.000264,0.000264
18,19,8519.99,0.47378,0.32486,0.83292,0.99038,0.98466,0.99254,0.94314,0.30988,0.22261,0.79180,0.000252,0.000252,0.000252


Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=8, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.05, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo/data_robust.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.25, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.6, hsv_v=0.5, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/kaggle/runs/detect/yolov8s_960_robust/weights/last.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=yolov8s_960_robust, nbs=64, nms=False, opset=None, opti

## 13. Validate clean and noisy accuracy


In [1]:
ROBUST_BEST_PATH = RUNS_DIR / "yolov8s_960_robust/weights/best.pt"
robust_best = YOLO(str(ROBUST_BEST_PATH))

clean_metrics = robust_best.val(
    data=str(DATA_YAML),
    imgsz=960,
    split="val",
    device=GPU_DEVICE,
    half=True,
    verbose=False,
)

noisy_metrics = robust_best.val(
    data=str(DATA_ROBUST_VAL_YAML),
    imgsz=960,
    split="val",
    device=GPU_DEVICE,
    half=True,
    verbose=False,
)

summary = pd.DataFrame([
    {
        "dataset": "clean_val",
        "mAP50": float(clean_metrics.box.map50),
        "mAP50_95": float(clean_metrics.box.map),
        "precision": float(clean_metrics.box.mp),
        "recall": float(clean_metrics.box.mr),
    },
    {
        "dataset": "noisy_val",
        "mAP50": float(noisy_metrics.box.map50),
        "mAP50_95": float(noisy_metrics.box.map),
        "precision": float(noisy_metrics.box.mp),
        "recall": float(noisy_metrics.box.mr),
    },
])
summary


NameError: name 'RUNS_DIR' is not defined

## 14. Confidence threshold sweep on noisy validation

For hidden noisy tests, lower confidence often improves recall. Pick the best confidence from this table.


In [ ]:
conf_results = []

for conf in [0.03, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30]:
    metrics = robust_best.val(
        data=str(DATA_ROBUST_VAL_YAML),
        imgsz=960,
        split="val",
        conf=conf,
        iou=0.6,
        device=GPU_DEVICE,
        half=True,
        verbose=False,
    )

    conf_results.append({
        "conf": conf,
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    })

df_conf = pd.DataFrame(conf_results).sort_values("mAP50_95", ascending=False).reset_index(drop=True)
BEST_CONF = float(df_conf.iloc[0]["conf"])
print("Best confidence on noisy val:", BEST_CONF)
df_conf


## 15. Benchmark prediction speed


In [ ]:
val_image_count = len(list(VAL_IMG_DIR.glob("*")))
start = time.time()
_ = robust_best.predict(
    source=str(VAL_IMG_DIR),
    imgsz=960,
    conf=BEST_CONF,
    iou=0.6,
    save=False,
    device=GPU_DEVICE,
    half=True,
    verbose=False,
)
end = time.time()

fps = val_image_count / (end - start)
print("Validation images:", val_image_count)
print("Prediction time sec:", end - start)
print("FPS:", fps)


## 16. Visual sanity check predictions


In [ ]:
predictions = robust_best.predict(
    source=str(ROBUST_VAL_IMG_DIR),
    imgsz=960,
    conf=BEST_CONF,
    iou=0.6,
    save=True,
    project=str(PREDICT_DIR),
    name="robust_noisy_val_predictions",
    exist_ok=True,
    device=GPU_DEVICE,
    half=True,
    verbose=False,
)

pred_folder = PREDICT_DIR / "robust_noisy_val_predictions"
pred_images = sorted(list(pred_folder.glob("*.jpg")) + list(pred_folder.glob("*.png")) + list(pred_folder.glob("*.jpeg")))
print("Prediction images found:", len(pred_images))

for img_path in pred_images[:5]:
    img = Image.open(img_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")
    plt.show()


## 17. Optional challenge/test prediction

Change `TEST_IMAGE_DIR` to the hidden/test image folder when you have it.


In [ ]:
TEST_IMAGE_DIR = Path("/kaggle/advanced/cv/test/images")

if TEST_IMAGE_DIR.exists():
    test_predictions = robust_best.predict(
        source=str(TEST_IMAGE_DIR),
        imgsz=960,
        conf=BEST_CONF,
        iou=0.6,
        augment=True,
        save=True,
        project=str(PREDICT_DIR),
        name="challenge_test_predictions",
        exist_ok=True,
        device=GPU_DEVICE,
        half=True,
        verbose=False,
    )
    print("Saved test predictions to:", PREDICT_DIR / "challenge_test_predictions")
else:
    print("TEST_IMAGE_DIR does not exist yet:", TEST_IMAGE_DIR)
    print("Set TEST_IMAGE_DIR to your challenge/test image folder before running this cell.")


## 18. Export final model and backup run folder


In [2]:
final_model_output = PREDICT_DIR / "yolov8s_960_robust_best.pt"
shutil.copy2(ROBUST_BEST_PATH, final_model_output)
print("Final robust model copied to:", final_model_output)

backup_zip = BACKUP_DIR / "yolov8s_960_robust_backup"
shutil.make_archive(str(backup_zip), "zip", root_dir=str(RUNS_DIR / "yolov8s_960_robust"))
print("Backup zip created:", str(backup_zip) + ".zip")

# Save metrics tables too
summary.to_csv(PREDICT_DIR / "robust_clean_vs_noisy_metrics.csv", index=False)
df_conf.to_csv(PREDICT_DIR / "robust_confidence_sweep.csv", index=False)
print("Saved metrics CSVs to:", PREDICT_DIR)


NameError: name 'PREDICT_DIR' is not defined

In [3]:
from pathlib import Path
import zipfile

best_pt = Path("/kaggle/runs/detect/yolov8s_960_robust/weights/best.pt")
zip_path = Path("/kaggle/working/yolov8s_960_robust_best.zip")

if not best_pt.exists():
    raise FileNotFoundError(f"Could not find: {best_pt}")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(best_pt, arcname="best.pt")

print("Created:", zip_path)
print("Size MB:", zip_path.stat().st_size / 1024 / 1024)

Created: /kaggle/working/yolov8s_960_robust_best.zip
Size MB: 19.770384788513184
